# Attack Classification Model Training

This notebook trains a classification model to categorize different types of attacks:
- **SQL Injection** (has_sql)
- **Script Injection** (has_script)
- **Path Traversal** (has_path_traversal)
- **Brute Force / Other** (attacks without specific patterns)

The model will be integrated into the backend to classify detected attacks in real-time.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import pickle
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)
import json

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

In [ ]:
# Load engineered features from v0.1.1 - sudah di-scale
features_path = Path('../data/processed/v0.1.1/features_engineered_dataset.csv')
df_features = pd.read_csv(features_path)

# Load raw data untuk mendapatkan path, query, body untuk pattern matching
raw_path = Path('../data/processed/v0.1.1/eda_cleaned_data.csv')
df_raw = pd.read_csv(raw_path)

print(f"Features dataset shape: {df_features.shape}")
print(f"Raw dataset shape: {df_raw.shape}")
print(f"\nFeatures columns: {df_features.columns.tolist()}")
print(f"Raw columns: {df_raw.columns.tolist()}")
print(f"\nLabel distribution in features:")
print(df_features['label'].value_counts())

In [ ]:
# Load raw data to get attack type information
raw_path = Path('../data/processed/final_dataset.csv')
df_raw = pd.read_csv(raw_path)

print(f"Raw dataset shape: {df_raw.shape}")
print(f"Raw columns: {df_raw.columns.tolist()}")
print(f"\nAttack type columns distribution:")
print(f"has_sql: {df_raw['has_sql'].value_counts()}")
print(f"has_script: {df_raw['has_script'].value_counts()}")
print(f"has_path_traversal: {df_raw['has_path_traversal'].value_counts()}")

In [ ]:
# Filter only attack records (label == 'Attack')
df_attacks = df_raw[df_raw['label'] == 'Attack'].copy()
print(f"Attack records: {len(df_attacks)} out of {len(df_raw)} ({len(df_attacks)/len(df_raw)*100:.1f}%)")

# Create classification labels based on attack patterns in path/query/body
def classify_attack_type(row):
    """
    Classify attack type berdasarkan pattern di path, query, dan body:
    - SQL Injection: UNION, SELECT, --comment, ;DROP, xp_
    - Script Injection: <script>, javascript:, onclick, onerror, eval()
    - Path Traversal: ../, ..\, ..%2f, %2e%2e
    - Command Injection: shell_exec, system(), exec(), backtick
    - Mixed/Other: jika multiple indicators atau tidak ada pattern
    """
    path = str(row.get('path', '')).lower()
    query = str(row.get('query', '')).lower()
    body = str(row.get('body', '')).lower()
    combined = path + query + body
    
    indicators = []
    
    # SQL Injection patterns
    sql_patterns = ['union', 'select', '--', '/*', 'drop', 'insert', 'update', 'delete', 
                   'xp_', 'sp_', 'cast(', 'convert(', 'substring(']
    if any(pattern in combined for pattern in sql_patterns):
        indicators.append('sql_injection')
    
    # Script/XSS patterns
    script_patterns = ['<script', 'javascript:', 'onerror=', 'onclick=', 'onload=', 
                      'eval(', 'src=', '<iframe', '<img', 'svg', 'alert(']
    if any(pattern in combined for pattern in script_patterns):
        indicators.append('script_injection')
    
    # Path Traversal patterns
    path_patterns = ['../', '..\\', '%2e%2e', '..%2f', '..%5c', '....', '/etc/', '/proc/']
    if any(pattern in combined for pattern in path_patterns):
        indicators.append('path_traversal')
    
    # Command Injection patterns
    cmd_patterns = ['shell_exec', 'system(', 'exec(', '`', '|cat', '; ls', '; rm', 'backtick']
    if any(pattern in combined for pattern in cmd_patterns):
        indicators.append('command_injection')
    
    # Classify based on indicators
    if len(indicators) > 1:
        return 'mixed_attack'
    elif len(indicators) == 1:
        return indicators[0]
    else:
        return 'other_attack'

df_attacks['attack_type'] = df_attacks.apply(classify_attack_type, axis=1)

print("\nAttack type distribution:")
print(df_attacks['attack_type'].value_counts())
print("\nAttack type percentages:")
print((df_attacks['attack_type'].value_counts(normalize=True) * 100).round(2))

In [ ]:
# Merge features dengan attack types
# Karena df_features sudah ter-scale, kita ambil yang corresponded dengan attack records

# Get indices dari attack records
attack_indices = df_raw[df_raw['label'] == 'Attack'].index.tolist()

# Get corresponding features
df_features_attacks = df_features.iloc[attack_indices].copy()
df_features_attacks['attack_type'] = df_attacks['attack_type'].values
df_features_attacks = df_features_attacks.reset_index(drop=True)

print(f"Features for attacks: {df_features_attacks.shape}")
print(f"\nFeatures columns: {df_features_attacks.columns.tolist()}")
print(f"\nAttack type distribution:")
print(df_features_attacks['attack_type'].value_counts())

# Encode attack types
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_features_attacks['attack_type_encoded'] = le.fit_transform(df_features_attacks['attack_type'])

# Create mapping
attack_type_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(f"\nAttack type encoding:")
for attack_type, code in sorted(attack_type_mapping.items(), key=lambda x: x[1]):
    print(f"  {code}: {attack_type}")

In [ ]:
# Select feature columns (exclude attack type labels)
excluded_cols = ['label', 'attack_type', 'attack_type_encoded']
feature_cols = [col for col in df_features_attacks.columns if col not in excluded_cols]

print(f"Number of features: {len(feature_cols)}")
print(f"Features: {feature_cols}")

X = df_features_attacks[feature_cols]
y = df_features_attacks['attack_type_encoded']

# Handle missing values
print(f"\nMissing values per column:")
missing = X.isnull().sum()[X.isnull().sum() > 0]
if len(missing) > 0:
    print(missing)
else:
    print("No missing values")

X = X.fillna(0)
X = X.replace([np.inf, -np.inf], 0)

# Features sudah di-scale dari v0.1.1, jadi tidak perlu scale lagi
# Tapi kita akan verify dan normalize jika diperlukan

# Split data untuk classification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"\nTrain set class distribution:")
print(y_train.value_counts())
print(f"\nTest set class distribution:")
print(y_test.value_counts())

In [ ]:
# Train XGBoost classifier
print("Training XGBoost classifier...")
clf_xgb = XGBClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    random_state=42,
    eval_metric='mlogloss',
    tree_method='hist'
)

# Data sudah di-scale, langsung train tanpa scaling lagi
clf_xgb.fit(X_train, y_train)

print("XGBoost model trained successfully")

# Also train RandomForest for comparison
print("\nTraining Random Forest classifier...")
clf_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

clf_rf.fit(X_train, y_train)

print("Random Forest model trained successfully")

In [ ]:
# Make predictions (data sudah di-scale)
y_pred_xgb = clf_xgb.predict(X_test)
y_pred_rf = clf_rf.predict(X_test)

# Evaluate XGBoost
print("=" * 60)
print("XGBoost Classifier Performance")
print("=" * 60)
xgb_accuracy = accuracy_score(y_test, y_pred_xgb)
xgb_precision = precision_score(y_test, y_pred_xgb, average='weighted', zero_division=0)
xgb_recall = recall_score(y_test, y_pred_xgb, average='weighted', zero_division=0)
xgb_f1 = f1_score(y_test, y_pred_xgb, average='weighted', zero_division=0)

print(f"Accuracy:  {xgb_accuracy:.4f}")
print(f"Precision: {xgb_precision:.4f}")
print(f"Recall:    {xgb_recall:.4f}")
print(f"F1-Score:  {xgb_f1:.4f}")

print("\nClassification Report (XGBoost):")
attack_type_reverse = {v: k for k, v in attack_type_mapping.items()}
y_test_labels = [attack_type_reverse[val] for val in y_test]
y_pred_xgb_labels = [attack_type_reverse[val] for val in y_pred_xgb]
print(classification_report(y_test_labels, y_pred_xgb_labels))

# Evaluate Random Forest
print("\n" + "=" * 60)
print("Random Forest Classifier Performance")
print("=" * 60)
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf, average='weighted', zero_division=0)
rf_recall = recall_score(y_test, y_pred_rf, average='weighted', zero_division=0)
rf_f1 = f1_score(y_test, y_pred_rf, average='weighted', zero_division=0)

print(f"Accuracy:  {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall:    {rf_recall:.4f}")
print(f"F1-Score:  {rf_f1:.4f}")

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_xgb = confusion_matrix(y_test, y_pred_xgb)
cm_rf = confusion_matrix(y_test, y_pred_rf)

attack_types = [attack_type_reverse[i] for i in sorted(attack_type_reverse.keys())]

sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=attack_types, yticklabels=attack_types)
axes[0].set_title('XGBoost Confusion Matrix')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=attack_types, yticklabels=attack_types)
axes[1].set_title('Random Forest Confusion Matrix')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

print("Confusion matrices displayed")

In [ ]:
# Feature importance dari XGBoost
feature_importance_xgb = pd.DataFrame({
    'feature': feature_cols,
    'importance': clf_xgb.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 15 Most Important Features (XGBoost):")
print(feature_importance_xgb.head(15))

# Visualize feature importance
fig, ax = plt.subplots(figsize=(12, 8))
feature_importance_xgb.head(15).plot(
    kind='barh',
    x='feature',
    y='importance',
    ax=ax,
    legend=False
)
ax.set_xlabel('Importance')
ax.set_ylabel('Feature')
ax.set_title('Top 15 Feature Importance - XGBoost Attack Classifier')
plt.tight_layout()
plt.show()

In [ ]:
# Pilih model terbaik (XGBoost lebih baik untuk classification)
best_model = clf_xgb
best_model_name = "XGBoost"

print(f"Selected model: {best_model_name}")
print(f"Accuracy: {xgb_accuracy:.4f}")

# Save model dan metadata
models_dir = Path('../models')
models_dir.mkdir(exist_ok=True)

# Save model
model_path = models_dir / 'attack-classification-v0.1.1.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(best_model, f)
print(f"\nModel saved to: {model_path}")

# Save metadata
metadata = {
    "model_name": best_model_name,
    "model_type": "MulticlassClassifier",
    "num_classes": len(attack_type_mapping),
    "class_mapping": attack_type_mapping,
    "class_reverse_mapping": attack_type_reverse,
    "num_features": len(feature_cols),
    "features": feature_cols,
    "accuracy": float(xgb_accuracy),
    "precision": float(xgb_precision),
    "recall": float(xgb_recall),
    "f1_score": float(xgb_f1),
    "attack_type_counts": df_attacks['attack_type'].value_counts().to_dict(),
    "training_data_size": len(X_train),
    "test_data_size": len(X_test),
    "training_date": pd.Timestamp.now().isoformat(),
}

metadata_path = models_dir / 'attack-classification-v0.1.1-metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"Metadata saved to: {metadata_path}")

print("\n" + "=" * 60)
print("Model Metadata:")
print("=" * 60)
for key, value in metadata.items():
    if key not in ['features', 'class_mapping', 'class_reverse_mapping']:
        print(f"{key}: {value}")
    elif key == 'class_mapping':
        print(f"{key}:")
        for k, v in value.items():
            print(f"  {v}: {k}")

## 7. Save Model to Backend

## 6. Feature Importance Analysis

## 5. Evaluate Model Performance

## 4. Build and Train Classification Model

In [ ]:
# Features sudah di-scale dari v0.1.1 pipeline, sehingga sudah siap untuk training
# Kita langsung gunakan data yang sudah di-scale
print("Data features sudah di-scale dari v0.1.1 pipeline")
print(f"Feature statistics:")
print(X_train.describe())

## 3. Prepare Data for Classification

## 2. Load and Explore Attack Detection Data

## 1. Import Required Libraries